# Aggregating data for contingency tables using Workbench


Federated analysis on contingency tables is relatively simple.
Counts are easy to federate: each TRE calculates their local count for some group, then these are aggregated by adding the counts together.
Each cell of a contingency table is a count, so the table can be federated by requesting these counts, and then statistical analyses can be performed on the aggregate.

```mermaid
graph TD
  subgraph 5s-TES
    sub(Submission layer)
    tre1(TRE 1)
    tre2(TREs ..n)
  end
  User -- Request counts --> sub
  sub -- Request counts --> tre1
  sub -- Request counts --> tre2
  tre1 -- counts --> agg(User)
  tre2 -- counts --> agg
  agg -- Sum counts --> Result
```

The following parameters will be used with `custom` mode of 5S TES Workbench:

| Field | value |
| ----- | ----- |
| Docker image| ghcr.io/health-informatics-uon/five-safes-tes-analytics-dev:sha-9ac04bc |
| Workdir | /app |
| Commands | --user-query=SELECT g.concept_name AS gender_name, r.concept_name AS race_name\nFROM \"OHDSIDemo\".person p\nJOIN \"OHDSIDemo\".concept g ON p.gender_concept_id = g.concept_id\nJOIN \"OHDSIDemo\".concept r ON p.race_concept_id = r.concept_id\nWHERE p.race_concept_id IN (8515, 8516, 8527)<br>--analysis=contingency_table<br>--output-filename=/outputs/output<br>--output-format=json |


## Part 1: Importing libraries and setting up Workbench

In [ ]:
from contingency_table_utils import aggregate_tables, read_contingency_table_from_json
from five_safes_tes_workbench.workbench import Workbench
from scipy.stats import chi2_contingency

wb = Workbench()

wb.validate(config_path="config.yml")

## Part 2: Building the TES task and submitting it

In [ ]:
_executors = [
                {
                    "image": "ghcr.io/health-informatics-uon/five-safes-tes-analytics-dev:sha-9ac04bc",
                    "command": [
                            """--user-query=SELECT g.concept_name AS gender_name, r.concept_name AS race_name
                            FROM \"OHDSIDemo\".person p
                            JOIN \"OHDSIDemo\".concept g ON p.gender_concept_id = g.concept_id
                            JOIN \"OHDSIDemo\".concept r ON p.race_concept_id = r.concept_id
                            WHERE p.race_concept_id IN (8515, 8516, 8527)""",
                            "--analysis=contingency_table",
                            "--output-filename=/outputs/output",
                            "--output-format=json"
                    ],
                    "workdir": "/app",
                },
         ]

# Build the TES task - should not be changed
wb.build_tes.custom(
    name="Workbench Demo - Contingency Table",
    description="Contingency Table",
    executors= _executors,
    
    outputs=[
        {
            "name": "Analysis Results Location",
            "description": "Analysis Results Location",
            "url": "s3://",
            "path": "/outputs",
            "type": "DIRECTORY"
    }
  ],
)

# Submit the TES task - should not be changed
wb.submit()

## Part 3: Fetching the outputs and reading them

In [ ]:
wb.fetch_outputs()

In [ ]:
# Remember to change the paths to the correct ones
tre1 = read_contingency_table_from_json("output/Nottingham TRE 01/id/output.json")
tre2 = read_contingency_table_from_json("output/Nottingham TRE 02/id/output.json")
tre1.data

## Part 4: Aggregating the tables
The `contingency_table` property organises this data into the format for statistical analyses.

In [ ]:
aggregate = aggregate_tables([tre1, tre2])
aggregate.contingency_table

## Part 5: Performing the chi-squared test
This format can be used for `scipy.stats` contingency table functions.

In [ ]:
chisq = chi2_contingency(aggregate.contingency_table)
print(f"The p-value for the chi-squared test is {chisq.pvalue:.3f}")